# Lab — EDA Univariate Analysis: Amazon UK Product Insights

This notebook answers the questions in `fisicangel/lab-eda-univariate` with reproducible code and a business-focused interpretation.

**Dataset location:** the notebook expects the dataset in a sibling folder named `Data_Amazon`:

```text
parent-folder/
├── Data_Amazon/
│   └── amz_uk_price_prediction_dataset.csv
└── lab-eda-univariate/
    └── lab-eda-univariate-solved.ipynb
```

The analysis is univariate: each section studies one variable at a time.

## Setup and data loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

DATA_FILENAME = "amz_uk_price_prediction_dataset.csv"
DATA_PATH = Path("..") / "Data_Amazon" / DATA_FILENAME

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Place the notebook inside the lab folder and keep Data_Amazon beside that folder."
    )

data = pd.read_csv(DATA_PATH)
print(f"Loaded {len(data):,} products from {DATA_PATH}")
display(data.head())

In [ ]:
required_columns = {"category", "price", "stars"}
missing_columns = required_columns.difference(data.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

overview = pd.DataFrame({
    "dtype": data[["category", "price", "stars"]].dtypes.astype(str),
    "missing": data[["category", "price", "stars"]].isna().sum(),
    "unique": data[["category", "price", "stars"]].nunique(),
})
display(overview)
print(f"Dataset shape: {data.shape[0]:,} rows × {data.shape[1]} columns")

## Part 1 — Understanding product categories

**Business question:** What are the most popular product categories on Amazon UK, and how do they compare in listing frequency?

### 1.1 Frequency table and top five categories

In [ ]:
category_frequency = (
    data["category"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .to_frame("count")
)
category_frequency["percentage"] = (
    category_frequency["count"] / len(data) * 100
)

display(category_frequency.head(10))
print("Top five most-listed categories:")
display(category_frequency.head(5))

**Finding.** `Sports & Outdoors` is the clear leader with **836,265 listings**, or about **34.22% of the entire dataset**. The next four categories each represent less than 1% individually: Beauty (19,312), Handmade Clothing, Shoes & Accessories (19,229), Bath & Body (19,092), and Birthday Gifts (18,978).

**Business interpretation.** The catalogue is highly concentrated in Sports & Outdoors. This category offers the greatest listing volume—and probably the strongest seller competition. The remaining catalogue has a long tail of much smaller categories.

### 1.2 Category visualizations

In [ ]:
top_20_categories = category_frequency.head(20).sort_values("count")

fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top_20_categories.index, top_20_categories["count"], color="steelblue")
ax.set(
    title="Twenty most-listed Amazon UK product categories",
    xlabel="Number of listings",
    ylabel="Category",
)
ax.xaxis.set_major_formatter(lambda x, pos: f"{x/1_000:,.0f}k")
plt.tight_layout()
plt.show()

In [ ]:
top_10_counts = category_frequency.head(10)["count"]
pie_values = pd.concat([
    top_10_counts,
    pd.Series({"All other categories": len(data) - top_10_counts.sum()}),
])

fig, ax = plt.subplots(figsize=(9, 9))
ax.pie(
    pie_values,
    labels=pie_values.index,
    autopct=lambda pct: f"{pct:.1f}%" if pct >= 2 else "",
    startangle=90,
    pctdistance=0.78,
)
ax.set_title("Share of all listings: top ten categories and the remainder")
plt.tight_layout()
plt.show()

**Chart interpretation.** No category exceeds 50% of all listings, so no single category holds an absolute majority. Nevertheless, Sports & Outdoors is disproportionately large at roughly one-third of the catalogue; the other leading categories are small by comparison. Including “All other categories” prevents the pie chart from exaggerating the top categories' shares.

## Part 2 — Delving into product pricing

**Business question:** How are products priced, and are particular price points or ranges more common?

### 2.1 Measures of central tendency

In [ ]:
price = data["price"].dropna()

price_centrality = pd.Series({
    "Mean": price.mean(),
    "Median": price.median(),
    "Mode": price.mode().iloc[0],
}, name="Price (£)")

display(price_centrality.to_frame())

**Finding.** The mean price is **£89.24**, the median is **£19.09**, and the mode is **£9.99**. A typical listing is therefore much cheaper than the arithmetic average.

**Business interpretation.** A relatively small number of very expensive products pull the mean upward. The median and mode better represent the price a shopper commonly encounters, while the mean alone would overstate the typical price point.

### 2.2 Measures of dispersion

In [ ]:
price_q1 = price.quantile(0.25)
price_q3 = price.quantile(0.75)
price_iqr = price_q3 - price_q1
price_lower_fence = price_q1 - 1.5 * price_iqr
price_upper_fence = price_q3 + 1.5 * price_iqr

price_dispersion = pd.Series({
    "Variance": price.var(),
    "Standard deviation": price.std(),
    "Minimum": price.min(),
    "Maximum": price.max(),
    "Range": price.max() - price.min(),
    "Q1": price_q1,
    "Q3": price_q3,
    "Interquartile range": price_iqr,
}, name="Price (£)")

display(price_dispersion.to_frame())

price_outliers = price[(price < price_lower_fence) | (price > price_upper_fence)]
print(f"IQR fences: £{price_lower_fence:,.2f} to £{price_upper_fence:,.2f}")
print(f"IQR-flagged outliers: {len(price_outliers):,} ({len(price_outliers) / len(price):.2%})")
print(f"99th percentile: £{price.quantile(0.99):,.2f}")

**Finding.** Price dispersion is extreme: variance is about **119,445.49**, standard deviation **£345.61**, range **£100,000**, and IQR **£36.00** (Q1 £9.99; Q3 £45.99). The IQR rule flags prices above £99.99 as outliers, representing about **13.41%** of listings. The 99th percentile is £1,408.82.

**Business interpretation.** The middle half of products occupies a compact £36 band, but the upper tail is exceptionally long. Pricing decisions should therefore use medians, percentiles, and category-level benchmarks rather than a catalogue-wide mean.

### 2.3 Price distribution and outliers

In [ ]:
price_p99 = price.quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(price, bins=100, color="steelblue", edgecolor="white")
axes[0].set_yscale("log")
axes[0].set(
    title="All prices (logarithmic count axis)",
    xlabel="Price (£)",
    ylabel="Number of listings (log scale)",
)

axes[1].hist(price[price <= price_p99], bins=60, color="darkorange", edgecolor="white")
axes[1].set(
    title=f"Prices up to the 99th percentile (£{price_p99:,.2f})",
    xlabel="Price (£)",
    ylabel="Number of listings",
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

sns.boxplot(x=price, ax=axes[0], color="lightsteelblue")
axes[0].set(title="All product prices", xlabel="Price (£)")

sns.boxplot(x=price[price <= price_p99], ax=axes[1], color="moccasin")
axes[1].set(
    title="Product prices up to the 99th percentile",
    xlabel="Price (£)",
)

plt.tight_layout()
plt.show()

**Chart interpretation.** The unfiltered plots are compressed near zero because a few extremely high prices stretch the x-axis. A logarithmic count axis keeps the long tail visible, while a 99th-percentile view reveals the structure of the vast majority of listings without silently deleting the extreme observations. The box plot confirms many high-price outliers and strong right skew.

## Part 3 — Unpacking product ratings

**Business question:** How do customers rate products, and what patterns appear?

The README refers to `rating`; in this dataset the field is named `stars`. A value of **0 stars** represents a listing with no displayed rating, not a genuine zero-star customer score. To remain faithful to the exercise, the raw-column results are reported first. A rated-only view is then added for a more useful business interpretation.

### 3.1 Measures of central tendency

In [ ]:
stars_all = data["stars"].dropna()
stars_rated = stars_all[stars_all > 0]

rating_centrality = pd.DataFrame({
    "All listings (includes 0 = unrated)": {
        "Count": len(stars_all),
        "Mean": stars_all.mean(),
        "Median": stars_all.median(),
        "Mode": stars_all.mode().iloc[0],
    },
    "Rated listings only": {
        "Count": len(stars_rated),
        "Mean": stars_rated.mean(),
        "Median": stars_rated.median(),
        "Mode": stars_rated.mode().iloc[0],
    },
})

display(rating_centrality)
print(f"Listings with a displayed rating: {len(stars_rated) / len(stars_all):.2%}")

**Finding.** Across the raw column, mean = **2.15**, median = **0.0**, and mode = **0.0**, because about half the listings are unrated. Among products with a displayed rating, mean = **4.32**, median = **4.4**, and mode = **4.5**.

**Business interpretation.** Customer sentiment among rated products is strongly positive. However, only about **49.84%** of listings have a displayed rating, so the raw 2.15 mean should not be described as poor customer satisfaction; it mostly reflects missing review coverage encoded as zero.

### 3.2 Measures of dispersion

In [ ]:
def dispersion_summary(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return pd.Series({
        "Variance": series.var(),
        "Standard deviation": series.std(),
        "Q1": q1,
        "Q3": q3,
        "Interquartile range": q3 - q1,
    })

rating_dispersion = pd.concat(
    {
        "All listings (includes 0 = unrated)": dispersion_summary(stars_all),
        "Rated listings only": dispersion_summary(stars_rated),
    },
    axis=1,
)
display(rating_dispersion)

**Finding.** In the raw column, variance = **4.82**, standard deviation = **2.19**, and IQR = **4.40**. For rated products, variance = **0.31**, standard deviation = **0.56**, and IQR = **0.50**.

**Business interpretation.** The raw data appears highly variable because it mixes unrated zeros with genuine ratings. Once unrated products are separated, customer ratings are tightly clustered around high values, indicating relatively consistent feedback.

### 3.3 Shape of the rating distribution

In [ ]:
rating_shape = pd.DataFrame({
    "All listings (includes 0 = unrated)": {
        "Skewness": stars_all.skew(),
        "Excess kurtosis": stars_all.kurt(),
    },
    "Rated listings only": {
        "Skewness": stars_rated.skew(),
        "Excess kurtosis": stars_rated.kurt(),
    },
})
display(rating_shape)

**Finding.** The raw column has skewness **0.081** and excess kurtosis **−1.926**, but these values describe a mixture of unrated zeros and real ratings—not a normal customer-rating distribution. Rated products alone are strongly left-skewed (about **−2.38**) with high positive excess kurtosis (about **9.78**).

**Interpretation.** Genuine ratings lean toward higher values, with most observations around 4–5 stars and a relatively small lower-rating tail. The bounded 1–5 scale, strong skew, and spike near 4.5 mean the distribution is not normal.

### 3.4 Rating visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rating_bins = np.arange(-0.05, 5.16, 0.1)
axes[0].hist(stars_all, bins=rating_bins, color="slateblue", edgecolor="white")
axes[0].set(
    title="Rating field for all listings",
    xlabel="Stars (0 means no displayed rating)",
    ylabel="Number of listings",
)

rated_bins = np.arange(0.95, 5.16, 0.1)
axes[1].hist(stars_rated, bins=rated_bins, color="seagreen", edgecolor="white")
axes[1].set(
    title="Displayed ratings only",
    xlabel="Stars",
    ylabel="Number of listings",
)

for ax in axes:
    ax.set_xlim(-0.1 if ax is axes[0] else 0.9, 5.1)

plt.tight_layout()
plt.show()

**Chart interpretation.** Zero is the most common value in the full field because many products have no displayed rating. Among rated products, **4.5 stars** is the most common value, and the distribution is concentrated toward the top of the scale.

## Executive summary

- **Category concentration:** Sports & Outdoors accounts for about 34.22% of all listings, far exceeding any other category. It is commercially important but likely highly competitive.
- **Price positioning:** The £89.24 mean is not representative of the typical product. The £19.09 median and £9.99 mode are more useful catalogue-wide reference points.
- **Price risk:** The distribution is strongly right-skewed; 13.41% of prices sit above the IQR upper fence of £99.99, and extreme products reach £100,000. Robust measures and category-specific comparisons are essential.
- **Rating quality:** Zero primarily means “unrated.” Among actually rated listings, the typical score is high (median 4.4; mode 4.5) and ratings are relatively consistent (IQR 0.5).
- **Business opportunity:** Improving review coverage could be as important as improving average ratings, because roughly half the catalogue has no displayed rating.